In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys

REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "main"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

In [ ]:
from recsys.io.artifact_store import ArtifactStore
from recsys.features.ingredients import (
    IngredientNormalizer,
    RareIngredientFilter,
    IngredientEncoder,
    QueryVectorizer,
    normalization_summary,
    rare_threshold_coverage_table,
)
from recsys.features.distance import JaccardDistance, CosineDistance
from recsys.contracts import (
    INGREDIENTS_OUTPUT_SCHEMA,
    INGREDIENTS_STAGE_SCHEMAS,
    LOADED_RECIPES_SCHEMA,
    validate,
)

store = ArtifactStore.from_config(CFG)

loaded = store.load("loaded")
loaded_manifest = loaded.manifest
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)

ing_cfg = CFG.section("ingredients")  # config/config.yaml `ingredients` block
modifiers = ing_cfg.get("modifiers")
min_count = int(ing_cfg.get("min_count", 10))
strategy = ing_cfg.get("encoding_strategy", "binary")

normalizer = IngredientNormalizer(modifiers=modifiers).fit(recipes)
recipes_clean = normalizer.transform(recipes)

summary_df, variants_df = normalization_summary(
    recipes_clean, modifiers=normalizer.modifiers_, top_n_examples=20
)
display(summary_df)
display(variants_df.head(15))

In [ ]:
# Note: the real pipeline fit happens in the 1.9 pipeline, on train data only.
# The output here is for catalogue exploration and for the 1.5 schema check.
coverage_df = rare_threshold_coverage_table(
    recipes_clean["ingredients_clean"],
    thresholds=(1, 2, 3, 5, 10, 20, 50, 100),
)
display(coverage_df)

rare_filter = RareIngredientFilter(min_count=min_count).fit(recipes_clean)
recipes_filtered = rare_filter.transform(recipes_clean)
print("Vocab size after min_count filter:", len(rare_filter.vocab_))

In [ ]:
binary_encoder = IngredientEncoder(strategy="binary").fit(recipes_filtered)
binary_matrix = binary_encoder.transform(recipes_filtered)

tfidf_encoder = IngredientEncoder(strategy="tfidf").fit(recipes_filtered)
tfidf_matrix = tfidf_encoder.transform(recipes_filtered)

assert len(binary_encoder.vocab_) == binary_matrix.shape[1]
assert len(tfidf_encoder.vocab_) == tfidf_matrix.shape[1]
print("Binary matrix:", binary_matrix.shape, "nnz:", binary_matrix.nnz)
print("TF-IDF matrix:", tfidf_matrix.shape, "nnz:", tfidf_matrix.nnz)

In [ ]:
qv = QueryVectorizer(
    normalizer=normalizer,
    vocab=binary_encoder.vocab_,
    strategy="binary",
)
sample_query = ["Fresh Garlic", "Chopped Onions", "Olive Oil", "Dragon Fruit Dust"]
q_vec, unknown_items = qv.vectorize(sample_query)
print("Unknown ingredients:", unknown_items)

# Jaccard and cosine distance check against the first 1000 recipes
j_dists = JaccardDistance().pairwise(q_vec, binary_matrix[:1000])[0]
c_dists = CosineDistance().pairwise(q_vec, binary_matrix[:1000])[0]
print("Min Jaccard dist (first 1000):", float(j_dists.min()))
print("Min Cosine dist  (first 1000):", float(c_dists.min()))

In [ ]:
ingredients_table = recipes_filtered[
    ["id", "ingredients", "ingredients_clean", "ingredients_filtered"]
]
validate(ingredients_table, INGREDIENTS_OUTPUT_SCHEMA)

version_dir = store.save(
    stage="ingredients",
    objects={
        "recipes_ingredients": ingredients_table,
        "normalization_summary": summary_df,
        "merged_variants": variants_df,
        "rare_threshold_coverage": coverage_df,
        "vocab": {"vocab": binary_encoder.vocab_},
        "binary_matrix": binary_matrix,
        "tfidf_matrix": tfidf_matrix,
        # vocab + idf of the tf-idf encoder: needed to rebuild QueryVectorizer(strategy="tfidf")
        "tfidf_idf": {"vocab": tfidf_encoder.vocab_, "idf": tfidf_encoder.idf_.tolist()},
    },
    inputs=[loaded],
    params={
        "min_count": min_count,
        "encoding_strategy": strategy,
        "n_modifiers": len(normalizer.modifiers_),
    },
    schemas=INGREDIENTS_STAGE_SCHEMAS,  # every table validated before writing
)
print("Saved ingredients artifacts to:", version_dir)